# EDA data

Kiểm tra xem tên cột & kiểu dữ liệu có giống nhau không

In [1]:
import pandas as pd
from pathlib import Path
import s3fs
import os
import pyarrow.parquet as pq

In [2]:
fs = s3fs.S3FileSystem()

data_dir = 's3://flezi-agent/nyc-taxi/'
profiles = []
list_files = fs.listdir(data_dir)
print(f'#files: {len(list_files)}')

for idx, item in enumerate(list_files):
    filename = item['Key'].split('/')[-1]
    filepath = os.path.join(data_dir, filename)

    if item['Key'].endswith('.parquet'):
        print(f'{idx}. {filename}')
        meta = pq.read_metadata(filepath)
        schema = pq.read_schema(filepath)
        profiles.append({
            "file": filename,
            "rows": meta.num_rows,
            "columns": schema.names,
            "types": [str(schema.field(c).type) for c in schema.names],
        })

profile_df = pd.DataFrame(profiles)
profile_df

#files: 205
1. yellow_tripdata_2009-01.parquet
2. yellow_tripdata_2009-02.parquet
3. yellow_tripdata_2009-03.parquet
4. yellow_tripdata_2009-04.parquet
5. yellow_tripdata_2009-05.parquet
6. yellow_tripdata_2009-06.parquet
7. yellow_tripdata_2009-07.parquet
8. yellow_tripdata_2009-08.parquet
9. yellow_tripdata_2009-09.parquet
10. yellow_tripdata_2009-10.parquet
11. yellow_tripdata_2009-11.parquet
12. yellow_tripdata_2009-12.parquet
13. yellow_tripdata_2010-01.parquet
14. yellow_tripdata_2010-02.parquet
15. yellow_tripdata_2010-03.parquet
16. yellow_tripdata_2010-04.parquet
17. yellow_tripdata_2010-05.parquet
18. yellow_tripdata_2010-06.parquet
19. yellow_tripdata_2010-07.parquet
20. yellow_tripdata_2010-08.parquet
21. yellow_tripdata_2010-09.parquet
22. yellow_tripdata_2010-10.parquet
23. yellow_tripdata_2010-11.parquet
24. yellow_tripdata_2010-12.parquet
25. yellow_tripdata_2011-01.parquet
26. yellow_tripdata_2011-02.parquet
27. yellow_tripdata_2011-03.parquet
28. yellow_tripdata_2011-

,file,rows,columns,types
0,yellow_tripdata_2009-01.parquet,14092413,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double..."
1,yellow_tripdata_2009-02.parquet,13380122,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double..."
2,yellow_tripdata_2009-03.parquet,14387371,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double..."
3,yellow_tripdata_2009-04.parquet,14294783,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double..."
4,yellow_tripdata_2009-05.parquet,14796313,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double..."
...,...,...,...,...
199,yellow_tripdata_2025-08.parquet,3574091,"[VendorID, tpep_pickup_datetime, tpep_dropoff_...","[int32, timestamp[us], timestamp[us], int64, d..."
200,yellow_tripdata_2025-09.parquet,4251015,"[VendorID, tpep_pickup_datetime, tpep_dropoff_...","[int32, timestamp[us], timestamp[us], int64, d..."
201,yellow_tripdata_2025-10.parquet,4428699,"[VendorID, tpep_pickup_datetime, tpep_dropoff_...","[int32, timestamp[us], timestamp[us], int64, d..."
202,yellow_tripdata_2025-11.parquet,4181444,"[VendorID, tpep_pickup_datetime, tpep_dropoff_...","[int32, timestamp[us], timestamp[us], int64, d..."


Các loại tên cột: Nhìn có vẻ same về ý nghĩa nên sẽ chọn 1 cái làm list tên cột cho tất cả

Lưu ý, có thể `pyarrow` sẽ chèn cột `__index_level_0__`. Bỏ qua thể loại này

In [3]:
cols_profile = profile_df['columns'].value_counts()
cols_profile

[VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, airport_fee]                        143
[VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, Airport_fee]                         23
[vendor_name, Trip_Pickup_DateTime, Trip_Dropoff_DateTime, Passenger_Count, Trip_Distance, Start_Lon, Start_Lat, Rate_Code, store_and_forward, End_Lon, End_Lat, Payment_Type, Fare_Amt, surcharge, mta_tax, Tip_Amt, Tolls_Amt, Total_Amt]                                                                              12
[VendorID, tpep_pickup_datetime, tpep_dropoff_dateti

Số lượng cột. Các cột này sẽ được semantic union lại, file nào không có cột nào thì cột đó sẽ chứa dữ liệu NULL

In [4]:
[len(i) for i in cols_profile.index]

[19, 19, 18, 20, 18, 19, 21]

List tên cột thống nhất: gồm 25 cột, sẽ được tạo trong table của DB. Tôi semantic union (bằng tay) các tên cột ở trên và thu được list này

In [5]:
LIST_COLS = ["vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count", "trip_distance", "rate_code_id", "store_and_fwd_flag", "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "pu_location_id", "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax", "surcharge", "improvement_surcharge", "tip_amount", "tolls_amount", "total_amount", "congestion_surcharge", "airport_fee", "cbd_congestion_fee"]
len(LIST_COLS)

25

Check sơ qua 7 file parquet có các kiểu column ở trên

In [6]:
profile_df['columns_transform'] = profile_df['columns'].apply(lambda list_cols: '|'.join(list_cols))
profile_df.head()

,file,rows,columns,types,columns_transform
0,yellow_tripdata_2009-01.parquet,14092413,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double...",vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_...
1,yellow_tripdata_2009-02.parquet,13380122,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double...",vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_...
2,yellow_tripdata_2009-03.parquet,14387371,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double...",vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_...
3,yellow_tripdata_2009-04.parquet,14294783,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double...",vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_...
4,yellow_tripdata_2009-05.parquet,14796313,"[vendor_name, Trip_Pickup_DateTime, Trip_Dropo...","[string, string, string, int64, double, double...",vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_...


In [9]:
df_names = [profile_df[profile_df['columns_transform'] == '|'.join(cols)]['file'].iloc[0] for cols in cols_profile.index]
df_names

['yellow_tripdata_2011-01.parquet',
 'yellow_tripdata_2023-02.parquet',
 'yellow_tripdata_2009-01.parquet',
 'yellow_tripdata_2025-01.parquet',
 'yellow_tripdata_2010-01.parquet',
 'yellow_tripdata_2010-02.parquet',
 'yellow_tripdata_2020-01.parquet']

In [11]:
dfs = [pd.read_parquet(os.path.join(data_dir, dfn))[:5] for dfn in df_names]

In [14]:
[df.info() for df in dfs]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               5 non-null      int64         
 1   tpep_pickup_datetime   5 non-null      datetime64[ns]
 2   tpep_dropoff_datetime  5 non-null      datetime64[ns]
 3   passenger_count        5 non-null      int64         
 4   trip_distance          5 non-null      float64       
 5   RatecodeID             5 non-null      int64         
 6   store_and_fwd_flag     0 non-null      object        
 7   PULocationID           5 non-null      int64         
 8   DOLocationID           5 non-null      int64         
 9   payment_type           5 non-null      int64         
 10  fare_amount            5 non-null      float64       
 11  extra                  5 non-null      float64       
 12  mta_tax                5 non-null      float64       
 13  tip_amoun

[None, None, None, None, None, None, None]

Có thể thấy, kiểu column thứ 5 và thứ 6 là giống nhau và không có cột `__index_level_0__` ở cuối. Do đó, tôi gộp 2 thể loại này thành 1, tăng số lượng file có kiểu cột như này thành 12

Tạo list mapping tên cột mới vào các file parquet để transform. Cái này cũng thực hiện bằng tay luôn. Có thể dùng AI (vì nó hiểu semantic) cho nhanh

In [18]:
col_types = pd.Series(cols_profile.index).apply(lambda cols: '|'.join(cols)).tolist()
col_types.pop(5)
print(len(col_types))
col_types

6


['VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee',
 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee',
 'vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|Trip_Distance|Start_Lon|Start_Lat|Rate_Code|store_and_forward|End_Lon|End_Lat|Payment_Type|Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt',
 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_am

In [ ]:
map1 = {
    "columns": 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
    }
}

map2 = {
    'columns':  'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map3 = {
    'columns': 'vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|Trip_Distance|Start_Lon|Start_Lat|Rate_Code|store_and_forward|End_Lon|End_Lat|Payment_Type|Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt',
    'mapping': {
        "vendor_name": "vendor_id",
        "Trip_Pickup_DateTime": "pickup_datetime",
        "Trip_Dropoff_DateTime": "dropoff_datetime",
        "Passenger_Count": "passenger_count",
        "Trip_Distance": "trip_distance",
        "Start_Lon": "pickup_longitude",
        "Start_Lat": "pickup_latitude",
        "Rate_Code": "rate_code_id",
        "store_and_forward": "store_and_fwd_flag",
        "End_Lon": "dropoff_longitude",
        "End_Lat": "dropoff_latitude",
        "Payment_Type": "payment_type",
        "Fare_Amt": "fare_amount",
        "Tip_Amt": "tip_amount",
        "Tolls_Amt": "tolls_amount",
        "Total_Amt": "total_amount",
    }
}

map4 = {
    'columns': 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map5 = {
    'columns':  'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

map6 = {
    'columns':  'vendorID|tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount',
    'mapping': {
        "vendorID": "vendor_id",
        "tpepPickupDateTime": "pickup_datetime",
        "tpepDropoffDateTime": "dropoff_datetime",
        "passengerCount": "passenger_count",
        "tripDistance": "trip_distance",
        "puLocationId": "pu_location_id",
        "doLocationId": "do_location_id",
        "startLon": "pickup_longitude",
        "startLat": "pickup_latitude",
        "endLon": "dropoff_longitude",
        "endLat": "dropoff_latitude",
        "rateCodeId": "rate_code_id",
        "storeAndFwdFlag": "store_and_fwd_flag",
        "paymentType": "payment_type",
        "fareAmount": "fare_amount",
        "mtaTax": "mta_tax",
        "improvementSurcharge": "improvement_surcharge",
        "tipAmount": "tip_amount",
        "tollsAmount": "tolls_amount",
        "totalAmount": "total_amount",
    }
}

maps = [map1, map2, map3, map4, map5, map6]

Mapping này sẽ được sử dụng trong quá trình transform data và đẩy vào database (data source)

Kiểm tra kiểu dữ liệu cho thấy một số kiểu dữ liệu bị conflict với nhau. Solution là đưa cho AI reasoning và chốt một kiểu dữ liệu tổng quát hơn cho mỗi cột bị conflict. Ví dụ, nếu 1 cột trong 2 files có kiểu dữ liệu `int` và `float` thì sẽ chọn `float`

In [9]:
profile_df['types'].value_counts()

[int64, timestamp[us], timestamp[us], int64, double, int64, string, int64, int64, int64, double, double, double, double, double, double, double, null, null]                            82
[int64, timestamp[us], timestamp[us], double, double, double, string, int64, int64, int64, double, double, double, double, double, double, double, double, double]                      29
[int32, timestamp[us], timestamp[us], int64, double, int64, large_string, int32, int32, int64, double, double, double, double, double, double, double, double, double]                  23
[int64, timestamp[us], timestamp[us], double, double, double, string, int64, int64, int64, double, double, double, double, double, double, double, double, null]                        20
[string, string, string, int64, double, double, double, double, double, double, double, string, double, double, double, double, double, double]                                         12
[int32, timestamp[us], timestamp[us], int64, double, int64, large

Sau khi sử dụng AI để check lại kiểu dữ liệu, sẽ thống nhất như sau

- Trong dataframe: Dùng làm target transform. Columns trong các files sẽ được cast về kiểu dữ liệu như này

    ```python
    {
        "vendor_id":             "object",
        "pickup_datetime":       "datetime64[ns]",
        "dropoff_datetime":      "datetime64[ns]",
        "passenger_count":       "float64",
        "trip_distance":         "float64",
        "rate_code_id":          "float64",
        "store_and_fwd_flag":    "object",
        "pickup_longitude":      "float64",
        "pickup_latitude":       "float64",
        "dropoff_longitude":     "float64",
        "dropoff_latitude":      "float64",
        "pu_location_id":        "object",
        "do_location_id":        "object",
        "payment_type":          "object",
        "fare_amount":           "float64",
        "extra":                 "float64",
        "mta_tax":               "float64",
        "surcharge":             "float64",
        "improvement_surcharge": "float64",
        "tip_amount":            "float64",
        "tolls_amount":          "float64",
        "total_amount":          "float64",
        "congestion_surcharge":  "float64",
        "airport_fee":           "float64",
        "cbd_congestion_fee":    "float64",
    }
    ```

- Trong database: Dùng để tạo bảng

    ```sql
    CREATE TABLE IF NOT EXISTS nyc_taxi (
        vendor_id               TEXT,
        pickup_datetime         TIMESTAMP,
        dropoff_datetime        TIMESTAMP,
        passenger_count         INT,
        trip_distance           FLOAT,
        rate_code_id            FLOAT,
        store_and_fwd_flag      TEXT,
        pickup_longitude        FLOAT,
        pickup_latitude         FLOAT,
        dropoff_longitude       FLOAT,
        dropoff_latitude        FLOAT,
        pu_location_id          TEXT,
        do_location_id          TEXT,
        payment_type            TEXT,
        fare_amount             FLOAT,
        extra                   FLOAT,
        mta_tax                 FLOAT,
        surcharge               FLOAT,
        improvement_surcharge   FLOAT,
        tip_amount              FLOAT,
        tolls_amount            FLOAT,
        total_amount            FLOAT,
        congestion_surcharge    FLOAT,
        airport_fee             FLOAT,
        cbd_congestion_fee      FLOAT
    );
    ```

Khi transform dữ liệu để đẩy vào db, cần thực hiện 2 việc:

- Đổi tên cột: Các file có tên cột khác với list thống nhất sẽ được đổi tên thành giống như list đó. Đối với các file bị thiếu cột nào (tất nhiên là sẽ thiếu) thì sẽ được tạo cột đó và đưa vào giá trị `NULL`
- Cast kiểu dữ liệu: Cast lại dữ liệu trong các cột để ghi xuống db. Sau khi đã cast xong 1 lượt, đổi lại kiểu dữ liệu cho toàn bộ df thành `object` và fill `None` (not `NaN`) vào các vị trí trống để ghi xuống db (nếu không làm thế thì nó sẽ ghi `NaN` thay vì `NULL` xuống db cho các kiểu dữ liệu dạng số)